In [1]:
using StaticArrays
using WaveGreen2D.Chebyshev: ChebyshevSeries, gradient, hessian, contains, clenshaw

In [5]:
const CONFIG = Dict(:threshold => 0.05, :max_iters => 500)

# CONFIG[key] = value # Perfectly fine and fast!

Dict{Symbol, Real} with 2 entries:
  :threshold => 0.05
  :max_iters => 500

In [7]:
CONFIG[:threshold]

0.05

In [1]:
mutable struct parameters
    depth::Float64
    frequency::Float64
end

In [2]:
const params = parameters(1.0, 2.0)

parameters(1.0, 2.0)

In [3]:
params.depth = 3.0

3.0

In [4]:
params.depth

3.0

In [ ]:
# cs_file = joinpath(@__DIR__, "chebyshev_series.jld2")
cs_file = joinpath("../src/chebyshev_series.jld2")
cs_jld2 = jldopen(cs_file)

const L₁_series = read(cs_jld2, "L₁_series")
const L₂_series = read(cs_jld2, "L₂_series")

close(cs_jld2)

In [11]:
function reduce_integrals(h::Real, K::Real)
    H = Float64(h*K)
    H̃ = log(H)

    x = SVector{3, Float64}(0.0, 0.0, H)
    x̃ = SVector{3, Float64}(0.0, 0.0, H̃)

    if contains(L₁_series[1], x)
        L₁ = clenshaw(L₁_series[1], H)
    elseif contains(L₁_series[2], x̃)
        L₁ = clenshaw(L₁_series[2], H̃)
    elseif contains(L₁_series[3], x̃)
        L₁ = clenshaw(L₁_series[3], H̃)
    else
        # TODO: Throw a warning that results might be innacurate due to H being out of bounds (H < 1e-2)
        L₁ = clenshaw(L₁_series[1], H)
    end

    if contains(L₂_series[1], x̃)
        L₂ = clenshaw(L₂_series[1], H̃)
    elseif contains(L₂_series[2], x̃)
        L₂ = clenshaw(L₂_series[2], H̃)
    elseif contains(L₂_series[3], x̃)
        L₂ = clenshaw(L₂_series[3], H̃)
    elseif contains(L₂_series[4], x̃)
        L₂ = clenshaw(L₂_series[4], H̃)
    else
        # TODO: Throw a warning that results might be innacurate due to H being out of bounds (H < 1e-2)
        L₂ = clenshaw(L₂_series[1], H̃)
    end
    
    return L₁, L₂
end

reduce_integrals (generic function with 1 method)

In [ ]:
function Λ₁(P::SVector{2, Float64}, Q::SVector{2, Float64})
    ξ, ζ = P
    x, z = Q

    R̄ = x - ξ
    R = abs(R̄)
    A = R/h

    v̄₁ = z - ζ
    v₁ = abs(v̄₁)
    B₁ = v₁/h

    u = SVector{2, Float64}(A, B₁)

    return L₁(u)
end


function ∇Λ₁(P::SVector{2, Float64}, Q::SVector{2, Float64})
    ξ, ζ = P
    x, z = Q

    R̄ = x - ξ
    R = abs(R̄)
    A = R/h
    dA_dx = sign(R̄)/h
    
    v̄₁ = z - ζ
    v₁ = abs(v̄₁)
    B₁ = v₁/h
    dB₁_dz = sign(v̄₁)/h

    u = SVector{2, Float64}(A, B₁)
    ∇u = SVector{2, Float64}(dA_dx, dB₁_dz)

    L, ∇ᵤL = gradient(L₁, u)

    # ∂L/∂x = ∂L/∂u ⋅ ∂u/∂x
    ∇L = ∇ᵤL .* ∇u
    
    return L, ∇L
end


function HΛ₁(P::SVector{2, Float64}, Q::SVector{2, Float64})
    ξ, ζ = P
    x, z = Q

    R̄ = x - ξ
    R = abs(R̄)
    A = R/h
    dA_dx = sign(R̄)/h
    
    v̄₁ = z - ζ
    v₁ = abs(v̄₁)
    B₁ = v₁/h
    dB₁_dz = sign(v̄₁)/h

    u = SVector{2, Float64}(A, B₁)
    ∇u = SVector{2, Float64}(dA_dx, dB₁_dz)
    ∇uᵈ = SMatrix{2, Float64}([dA_dx 0.0; 0.0 dB₁_dz])

    L, ∇ᵤL, HᵤL = hessian(L₁, u)

    # ∂L/∂x = ∂L/∂u ⋅ ∂u/∂x
    ∇L = ∇ᵤL .* ∇u

    # ∂²L/∂x² = ∂²L/∂u² ⋅ (∂u/∂x)²
    HL = ∇uᵈ * HᵤL * ∇uᵈ
    
    return L, ∇L, HL
end;